In [1]:
from torch import tensor
from torchvision import datasets, transforms
import numpy as np
import sklearn
from sklearn.model_selection import StratifiedKFold
import logging
import math
import random
from sklearn.mixture import GaussianMixture
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.stats import kstest, anderson_ksamp, cumfreq, ks_2samp, cramervonmises, chisquare, entropy, wasserstein_distance

In [2]:
from keras.preprocessing.image import ImageDataGenerator

In [22]:
def get_data(dataset_type, path=None):
    
    if dataset_type == "MNIST":
        train, test = mnist.load_data()
    elif dataset_type == "CIFAR10":
        train, test = cifar10.load_data()
        
    train_x, train_y = from_numpy(train[0]), from_numpy(train[1])
    test_x, test_y = from_numpy(test[0]), from_numpy(test[1])
    train, test = (train_x, train_y), (test_x, test_y)
        
    return train, test

In [4]:
def log_class_counts(y_train, subset_ID_map, log=False):
    cls_counts = {}

    for subset_i, ID in subset_ID_map.items():
        unq, unq_cnt = np.unique(y_train[ID], return_counts=True)
        tmp = {unq[i]: unq_cnt[i] for i in range(len(unq))}
        cls_counts[subset_i] = tmp

    if log:
        logging.debug('Label distributions: %s' % str(cls_counts))

    return cls_counts

In [5]:
def map_to_prob(y_train, subset_map):
    counts = log_class_counts(y_train, subset_map)

    values = [np.array([counts.get(k).get(key) for key in counts.get(k)]) for k in counts]
    probs = [values[j] / values[j].sum() for j in range(len(values))]

    return probs

In [23]:
def partition_homo_skf(train, n_clients, alpha=0):
    subset_ID_map = {}
    x_train, y_train = train[0], train[1]
    # n_train = y_train.shape[0]
    skf = StratifiedKFold(n_splits=n_clients, shuffle=True, random_state=42)
    subsets = []
    for train_ID, test_ID in skf.split(train, y_train):
        subsets.append(test_ID)
    for j in range(n_clients):
        # np.random.shuffle(subsets[j])
        subset_ID_map[j] = subsets[j]
    return subset_ID_map

In [24]:
def partition_hetero_dir(train, n_clients, alpha):
    x_train, y_train = train[0], train[1]
    
    min_size = 0
    # classes
    K = len(y_train.unique())
    # data points
    N = y_train.shape[0]
    subset_ID_map = {}

    while min_size < 10:
        subset_ID_list = [[] for _ in range(n_clients)]
        for k in range(K):
            ids_k = np.where(y_train == k)[0]
            np.random.shuffle(ids_k)
            proportions = np.random.dirichlet(np.repeat(alpha, n_clients))
            proportions = np.array(
                [p * (len(ids_j) < N / n_clients) for p, ids_j in zip(proportions, subset_ID_list)])
            proportions = proportions / proportions.sum()
            proportions = (np.cumsum(proportions) * len(ids_k)).astype(int)[:-1]
            subset_ID_list = [ids_j + ids.tolist() for ids_j, ids in
                              zip(subset_ID_list, np.split(ids_k, proportions))]
            min_size = min([len(ids_j) for ids_j in subset_ID_list])

    for j in range(n_clients):
        np.random.shuffle(subset_ID_list[j])
        subset_ID_map[j] = subset_ID_list[j]

    return subset_ID_map

In [26]:
def partition_hetero_gaussian(train, n_clients, alpha, bootstrap=False):
    x_train, y_train = train[0], train[1]
    min_size = 0
    # classes
    K = len(set(y_train.tolist()))
    # data points
    N = y_train.shape[0]
    subset_ID_map = {}

    def gaussian_pdf(x, mu, sig):
        return np.exp(- np.power(x - mu, 2.) / (2 * np.power(sig, 2.))) / (sig * math.sqrt(2 * math.pi))

    labels = list(set(y_train.tolist()))
    mu = 5
    sig = alpha
    proportions = np.array([gaussian_pdf(x, mu, sig) for x in labels])

    if bootstrap:
        # make sure the most frequent label always gets selected
        norm_const = 1 / max(proportions)
        proportions = [item * norm_const for item in proportions]
        prob_map = {labels[i]: proportions[i] for i in range(len(labels))}

        def shuffle_labels(map):
            max_item = max(map, key=map.get)
            max_val = map.get(max_item)
            del map[max_item]
            keys = list(map.keys())
            values = list(map.values())
            random.shuffle(values)
            new_map = {keys[i]: values[i] for i in range(len(keys))}
            new_map.update({max_item:max_val})
            return new_map

        prob_map = shuffle_labels(prob_map)

        subset_ID_list = [[] for _ in range(n_clients)]

        for k in range(K):
            sample = []

            for ID, x in enumerate(y_train.tolist):
                if random.uniform(0, 1) < float(prob_map.get(x)):
                    sample.append(ID)

            subset_ID_list[k] = sample

    else:
        while min_size < 10:
            subset_ID_list = [[] for _ in range(n_clients)]
            for k in range(K):
                ids_k = np.where(y_train == k)[0]
                np.random.shuffle(ids_k)
                proportions = np.array([gaussian_pdf(x, mu, sig) for x in labels])
                proportions = np.array(
                    [p * (len(ids_j) < N / n_clients) for p, ids_j in zip(proportions, subset_ID_list)])
                if proportions.sum()!=0:
                    proportions = proportions / proportions.sum()
                proportions = (np.cumsum(proportions) * len(ids_k)).astype(int)[:-1]
                subset_ID_list = [ids_j + ids.tolist() for ids_j, ids in
                                  zip(subset_ID_list, np.split(ids_k, proportions))]
                min_size = min([len(ids_j) for ids_j in subset_ID_list])

    subset_ID_map = {i: subset_ID_list[i] for i in range(n_clients)}

    return subset_ID_map

In [27]:
def partition_quantity_based(train, n_clients, alpha):
    # sorted = pd.concat([y for x, y in data.groupby(0)]).reset_index().drop(columns=['index'])
    x, y = train[0], train[1]    
    K = len(y.unique())
    # subsets_pure = {i: np.where(y == i)[0] for i in range(10)}
    subsets_pure = np.concatenate([np.where(y == i)[0] for i in range(K)])

    ids = np.arange(y.shape[0])
    batch_ids = np.array_split(ids, n_clients*alpha)
    minibatches = []

    for i in range(n_clients*alpha):
        batch_i = [subsets_pure[j] for j in batch_ids[i]]
        minibatches.append(batch_i)

    minibatches = np.array(minibatches)

    ids_mini = np.random.permutation(range(n_clients*alpha))
    subset_indices = np.array_split(ids_mini, n_clients)

    clients = {}

    for index in subset_indices:
        client_i = []
        for i in index:
            client_i.append(minibatches[i])
        client_i = np.concatenate(client_i)
        clients.update({i:np.array(client_i)})

    return clients


In [10]:
def quantity_efficient(path, n_clients, alpha):
    data = datasets.MNIST(root=path, train=True, download=True, transform=None)
    x, y = data.data, data.targets.numpy()
    K = len(set(y.tolist()))
    ids = np.arange(y.shape[0])
    subsets_pure = np.concatenate([np.where(y == i)[0] for i in range(K)])
    minibatches = np.array_split(subsets_pure, n_clients * alpha)
    clients = {i: np.concatenate(minibatches[i::n_clients]) for i in range(n_clients)}
    return clients

In [36]:
def partition(dataset_type, type, n_clients, alpha):
    train, test = get_data(dataset_type, "D:/datasets")
    
    partition_funcs = {
        "homo" : partition_homo_skf,
        "hetero-dir": partition_hetero_dir,
        "hetero-gaussian": partition_hetero_gaussian,
        "quant": partition_quantity_based
    }
    
    try:
        partition_func = partition_funcs[type]
    except KeyError:
        raise ValueError(f"Invalid mode: {type}")

    return partition_func(train, n_clients, alpha)

In [32]:
def vis_datasets(path, epoch, y_train, subset_ID_map, mode_plot, mode_partitioning, alpha, save=False):
    counts = log_class_counts(y_train, subset_ID_map)

    values = [np.array([counts.get(k).get(key) for key in counts.get(k)]) for k in counts]
    values_normalized = [values[j] / values[j].sum() for j in range(len(values))]

    n_clients = len(subset_ID_map)

    title_formats = {
        "hetero-dir": "A distribution-based heterogeneous partitioning X~Dir({alpha}) with {n_clients} subsets",
        "hetero-gaussian": "A distribution-based Gaussian heterogeneous partitioning σ={alpha} with {n_clients} subsets",
        "homo": "A homogeneous partitioning with {n_clients} subsets",
    }
    main_title = title_formats.get(mode_partitioning, "")
    main_title = main_title.format(alpha=alpha, n_clients=n_clients)

    subtitle_formats = {
        "hetero_dir": "α_{epoch}={alpha}",
        "hetero-gaussian": "σ_{epoch}={alpha}",
        "homo" : ""
    }

    subtitle = subtitle_formats.get(mode_partitioning, "")
    subtitle = subtitle.format(epoch=epoch, alpha=alpha)

    if mode_plot == "heatmap":
        ax = sns.heatmap(pd.DataFrame(values_normalized), vmin=0, vmax=1, cmap=sns.cm.rocket_r)
        ax.set(xlabel="Labels", ylabel="Clients", title=main_title)
        plt.show()
    elif mode_plot == "histogram":
        dim_x = 2
        dim_y = 5
        K = len(set(y_train.tolist()))
        fig, axes = plt.subplots(dim_y, dim_x)
        # TODO: orient hard-coded values around 'K'
        for j in range(len(values)):
            plt.figure(figsize=(5, 3), dpi=300)
            plt.hist(values[j], [(i - 0.5) / 2 for i in range(2*K)], label="Sampled dist")
            x = np.arange(-0.5, 9.5, 0.1)
            plt.xticks([i for i in range(K)])
            plt.xlabel("Label")
            plt.xlim([-1, 10])
            plt.ylabel("Entries")
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.show()

In [13]:
def tensor_to_csv(x_train, y_train, subset_map, epoch_num):
    for key in subset_map:
        x, y = x_train[subset_map.get(key)], y_train[subset_map.get(key)]
        df_x = pd.DataFrame(x.tolist())
        df_y = pd.DataFrame(y.tolist())
        df_x['targets'] = df_y
        df_x.rename(columns={0: 'data', "targets": "targets"})
        df_x.to_csv(f'subsets/epoch_{epoch_num}_subset_{key + 1}.csv', index=False)

In [ ]:
def getDivs(N):
    factors = {1}
    maxP  = int(N**0.5)
    p,inc = 2,1
    while p <= maxP:
        while N%p==0:
            factors.update([f*p for f in factors])
            N //= p
            maxP = int(N**0.5)
        p,inc = p+inc,2
    if N>1:
        factors.update([f*N for f in factors])
    return sorted(factors)  

In [34]:
from keras import models, layers, optimizers
from keras.datasets import mnist, cifar10
from torch import from_numpy

In [17]:
def create_model(alpha, train):
    model = models.Sequential()
    model.add(layers.Flatten(input_shape=train[0].shape[1:3]))
    model.add(layers.Dense(units=256, activation='relu'))
    model.add(layers.Dense(units=128, activation='relu'))
    # First hidden layer
    model.add(layers.Dropout(rate=0.4))
    p = len(set(train[1].tolist()))
    model.add(layers.Dense(units=p, activation='softmax'))
    model.compile(optimizer=optimizers.Adam(learning_rate=alpha),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [18]:
def custom_model(train, lrate, hidden_units, act_fct_hidden, act_fct_output, drop_rate, metrics):
    model = models.Sequential()
    model.add(layers.Flatten(input_shape=train[0].shape[1:3]))
    for units in hidden_units:
        model.add(layers.Dense(units=units, activation=act_fct_hidden))
        model.add(layers.Dropout(rate=drop_rate))
    p = len(set(train[1].tolist()))
    model.add(layers.Dense(units=p, activation=act_fct_output))
    model.compile(optimizer=optimizers.Adam(learning_rate=lrate),
                  loss='sparse_categorical_crossentropy',
                  metrics=metrics)
    return model

In [56]:
def train_model(model, train_data, train_targets, epochs, batch_size=None, validation_split=0.2):
    history = model.fit(x=train_data, y=train_targets, batch_size=batch_size,
                        epochs=epochs, shuffle=True, validation_split=validation_split)
    epochs=history.epoch
    hist = pd.DataFrame(history.history)
    return epochs, hist

In [20]:
def plot_curve(epochs, hist, list_of_metrics):
  plt.figure()
  plt.xlabel("Epoch")
  plt.ylabel("Value")

  for m in list_of_metrics:
    x = hist[m]
    plt.plot(epochs[1:], x[1:], label=m)

  plt.legend()

In [ ]:
def run_experiment(dataset_type, type, n_clients, alpha, lrate, epochs, run_info=None):
    dataset_loaders = {
        "MNIST": mnist.load_data,
        "CIFAR10": cifar10.load_data,
    }

    try:
        loader = dataset_loaders[dataset_type]
    except KeyError:
        raise ValueError(f"Invalid mode: {dataset_type}")
        
    train, test = loader()
        
#    if dataset_type == "MNIST":
 #       train, test = mnist.load_data()
  #  elif dataset_type == "CIFAR10":
   #     train, test = cifar10.load_data()
        
    subset_map = partition(dataset_type, type, n_clients, alpha)
    trained_models = []
    hist = []
    accuracies = []
    for j in range(len(subset_map)):
        subset_j_data = train[0][subset_map[j]]
        subset_j_targets = train[1][subset_map[j]]
        model_j = create_model(lrate, train)
        epochs_j, hist_j = train_model(model_j, subset_j_data, subset_j_targets, epochs, validation_split=0.2)
        foldername = f'run{run_info}'
        filename = f'run_{run_info}_client_{j}.h5'
        pathname = "models/" + foldername + '/' + filename
        model_j.save(filename)
        print("Saving "+ filename)
        trained_models.append(model_j)
        hist.append(hist_j)        
        loss, accuracy = model.evaluate(x=test[0], y=test[1], batch_size=4000)
        accuracies.append(accuracy)
    
    return accuracies              

In [ ]:
def run_multi_experiment(dataset_type, type, n_clients, alpha, lrate, epochs):
    dict_accuracies = {}
    accuracies_mean = []
    accuracies_std = []
    
    for j, alpha_j in enumerate(alpha):
        accuracies_j = run_experiment(dataset_type, type, n_clients, alpha_j, lrate, epochs)
        dict_accuracies.update({j:accuracies_j})
        mean_j = np.array(accuracies_j).mean()
        accuracies_mean.append(mean_j)
        std_j = np.array(accuracies_j).std()
        accuracies_std.append(std_j)
    
    for j in range(len(accuracies_mean)):
        plt.figure(figsize=(5, 3), dpi=300)
        plt.plot(np.arange(accuracies_mean), accuracies_mean)
        plt.fill_between(np.arange(accuracies_mean), accuracies_mean - accuracies_std, 
                         accuracies_mean + accuracies_std,color='gray', alpha=0.2)
        plt.grid(linestyle='-', linewidth=0.5)
        # plt.suptitle(main_title, fontsize='small')
        plt.show()    
        
    return dict_accuracies              

In [ ]:
run_multi_experiment("MNIST", "hetero-dir", 10, np.arange(0.5, 10.5, 1).tolist(), 0.0003, 50)

Epoch 1/50
151/151 [==============================] - 2s 7ms/step - loss: 22.2039 - accuracy: 0.6192 - val_loss: 2.6107 - val_accuracy: 0.8567
Epoch 2/50
151/151 [==============================] - 1s 5ms/step - loss: 3.6114 - accuracy: 0.7640 - val_loss: 1.3037 - val_accuracy: 0.8550
Epoch 3/50
151/151 [==============================] - 1s 4ms/step - loss: 1.6071 - accuracy: 0.7601 - val_loss: 1.0368 - val_accuracy: 0.8285
Epoch 4/50
151/151 [==============================] - 1s 4ms/step - loss: 1.0191 - accuracy: 0.7866 - val_loss: 0.8537 - val_accuracy: 0.8600
Epoch 5/50
151/151 [==============================] - 1s 4ms/step - loss: 0.8287 - accuracy: 0.8032 - val_loss: 0.7652 - val_accuracy: 0.8683
Epoch 6/50
151/151 [==============================] - 1s 4ms/step - loss: 0.6601 - accuracy: 0.8299 - val_loss: 0.6987 - val_accuracy: 0.8782
Epoch 7/50
151/151 [==============================] - 1s 4ms/step - loss: 0.5658 - accuracy: 0.8444 - val_loss: 0.6748 - val_accuracy: 0.8865
Epoch

140/140 [==============================] - 1s 6ms/step - loss: 0.4777 - accuracy: 0.8991 - val_loss: 0.6173 - val_accuracy: 0.9114
Epoch 8/50
140/140 [==============================] - 1s 6ms/step - loss: 0.3926 - accuracy: 0.9022 - val_loss: 0.6511 - val_accuracy: 0.9167
Epoch 9/50
140/140 [==============================] - 1s 5ms/step - loss: 0.3364 - accuracy: 0.9116 - val_loss: 0.6172 - val_accuracy: 0.9239
Epoch 10/50
140/140 [==============================] - 1s 6ms/step - loss: 0.3671 - accuracy: 0.9087 - val_loss: 0.5872 - val_accuracy: 0.9212
Epoch 11/50
140/140 [==============================] - 1s 6ms/step - loss: 0.2594 - accuracy: 0.9241 - val_loss: 0.5422 - val_accuracy: 0.9212
Epoch 12/50
140/140 [==============================] - 1s 5ms/step - loss: 0.2836 - accuracy: 0.9244 - val_loss: 0.6225 - val_accuracy: 0.9302
Epoch 13/50
140/140 [==============================] - 1s 6ms/step - loss: 0.2492 - accuracy: 0.9286 - val_loss: 0.5529 - val_accuracy: 0.9355
Epoch 14/50
1

Epoch 14/50
160/160 [==============================] - 1s 9ms/step - loss: 0.1352 - accuracy: 0.9637 - val_loss: 0.3535 - val_accuracy: 0.9560
Epoch 15/50
160/160 [==============================] - 1s 9ms/step - loss: 0.1249 - accuracy: 0.9704 - val_loss: 0.3960 - val_accuracy: 0.9592
Epoch 16/50
160/160 [==============================] - 1s 8ms/step - loss: 0.1533 - accuracy: 0.9609 - val_loss: 0.4158 - val_accuracy: 0.9537
Epoch 17/50
160/160 [==============================] - 1s 7ms/step - loss: 0.1466 - accuracy: 0.9676 - val_loss: 0.3616 - val_accuracy: 0.9545
Epoch 18/50
160/160 [==============================] - 1s 8ms/step - loss: 0.1632 - accuracy: 0.9647 - val_loss: 0.4170 - val_accuracy: 0.9529
Epoch 19/50
160/160 [==============================] - 1s 8ms/step - loss: 0.1307 - accuracy: 0.9692 - val_loss: 0.3485 - val_accuracy: 0.9623
Epoch 20/50
160/160 [==============================] - 1s 8ms/step - loss: 0.1830 - accuracy: 0.9635 - val_loss: 0.3708 - val_accuracy: 0.9576

194/194 [==============================] - 1s 6ms/step - loss: 0.2379 - accuracy: 0.9296 - val_loss: 0.4062 - val_accuracy: 0.9471
Epoch 21/50
194/194 [==============================] - 1s 7ms/step - loss: 0.2303 - accuracy: 0.9323 - val_loss: 0.4377 - val_accuracy: 0.9458
Epoch 22/50
194/194 [==============================] - 1s 7ms/step - loss: 0.2073 - accuracy: 0.9410 - val_loss: 0.4248 - val_accuracy: 0.9484
Epoch 23/50
194/194 [==============================] - 1s 5ms/step - loss: 0.2195 - accuracy: 0.9381 - val_loss: 0.4253 - val_accuracy: 0.9446
Epoch 24/50
194/194 [==============================] - 1s 6ms/step - loss: 0.2165 - accuracy: 0.9379 - val_loss: 0.3689 - val_accuracy: 0.9446
Epoch 25/50
194/194 [==============================] - 1s 6ms/step - loss: 0.2346 - accuracy: 0.9318 - val_loss: 0.4507 - val_accuracy: 0.9471
Epoch 26/50
194/194 [==============================] - 1s 5ms/step - loss: 0.1905 - accuracy: 0.9421 - val_loss: 0.4611 - val_accuracy: 0.9497
Epoch 27/50

186/186 [==============================] - 1s 4ms/step - loss: 0.0743 - accuracy: 0.9940 - val_loss: 0.1485 - val_accuracy: 0.9919
Epoch 27/50
186/186 [==============================] - 1s 4ms/step - loss: 0.0679 - accuracy: 0.9950 - val_loss: 0.1625 - val_accuracy: 0.9913
Epoch 28/50
186/186 [==============================] - 1s 4ms/step - loss: 0.0737 - accuracy: 0.9960 - val_loss: 0.0600 - val_accuracy: 0.9946
Epoch 29/50
186/186 [==============================] - 1s 5ms/step - loss: 0.0491 - accuracy: 0.9961 - val_loss: 0.0435 - val_accuracy: 0.9966
Epoch 30/50
186/186 [==============================] - 1s 5ms/step - loss: 0.0403 - accuracy: 0.9960 - val_loss: 0.1402 - val_accuracy: 0.9933
Epoch 31/50
186/186 [==============================] - 1s 5ms/step - loss: 0.0246 - accuracy: 0.9966 - val_loss: 0.0776 - val_accuracy: 0.9960
Epoch 32/50
186/186 [==============================] - 1s 5ms/step - loss: 0.0469 - accuracy: 0.9961 - val_loss: 0.0330 - val_accuracy: 0.9966
Epoch 33/50

...

Epoch 28/50
154/154 [==============================] - 1s 5ms/step - loss: 0.3051 - accuracy: 0.9091 - val_loss: 0.7155 - val_accuracy: 0.9015
Epoch 29/50
154/154 [==============================] - 1s 5ms/step - loss: 0.2761 - accuracy: 0.9245 - val_loss: 0.6637 - val_accuracy: 0.9024
Epoch 30/50
154/154 [==============================] - 1s 5ms/step - loss: 0.2583 - accuracy: 0.9298 - val_loss: 0.7814 - val_accuracy: 0.8942
Epoch 31/50
154/154 [==============================] - 1s 5ms/step - loss: 0.2717 - accuracy: 0.9225 - val_loss: 0.8250 - val_accuracy: 0.8991
Epoch 32/50
154/154 [==============================] - 1s 5ms/step - loss: 0.2705 - accuracy: 0.9235 - val_loss: 0.7955 - val_accuracy: 0.9007
Epoch 33/50
154/154 [==============================] - 1s 5ms/step - loss: 0.2151 - accuracy: 0.9349 - val_loss: 0.7819 - val_accuracy: 0.9064
Epoch 34/50
154/154 [==============================] - 1s 5ms/step - loss: 0.2351 - accuracy: 0.9359 - val_loss: 0.9059 - val_accuracy: 0.9064

164/164 [==============================] - 1s 6ms/step - loss: 0.2738 - accuracy: 0.9181 - val_loss: 0.7355 - val_accuracy: 0.9097
Epoch 35/50
164/164 [==============================] - 1s 6ms/step - loss: 0.2770 - accuracy: 0.9234 - val_loss: 0.6764 - val_accuracy: 0.9051
Epoch 36/50
164/164 [==============================] - 1s 5ms/step - loss: 0.2490 - accuracy: 0.9271 - val_loss: 0.6888 - val_accuracy: 0.9151
Epoch 37/50
164/164 [==============================] - 1s 5ms/step - loss: 0.2438 - accuracy: 0.9271 - val_loss: 0.6539 - val_accuracy: 0.9158
Epoch 38/50
164/164 [==============================] - 1s 5ms/step - loss: 0.2592 - accuracy: 0.9286 - val_loss: 0.6791 - val_accuracy: 0.9097
Epoch 39/50
164/164 [==============================] - 1s 5ms/step - loss: 0.2365 - accuracy: 0.9273 - val_loss: 0.7034 - val_accuracy: 0.9128
Epoch 40/50
164/164 [==============================] - 1s 5ms/step - loss: 0.2311 - accuracy: 0.9351 - val_loss: 0.7538 - val_accuracy: 0.9143
Epoch 41/50

136/136 [==============================] - 1s 5ms/step - loss: 0.2691 - accuracy: 0.9345 - val_loss: 0.9429 - val_accuracy: 0.9073
Epoch 41/50
136/136 [==============================] - 1s 5ms/step - loss: 0.2584 - accuracy: 0.9281 - val_loss: 0.7737 - val_accuracy: 0.9164
Epoch 42/50
136/136 [==============================] - 1s 5ms/step - loss: 0.2315 - accuracy: 0.9322 - val_loss: 0.8335 - val_accuracy: 0.9192
Epoch 43/50
136/136 [==============================] - 1s 5ms/step - loss: 0.2250 - accuracy: 0.9393 - val_loss: 0.8089 - val_accuracy: 0.9155
Epoch 44/50
136/136 [==============================] - 1s 4ms/step - loss: 0.1982 - accuracy: 0.9469 - val_loss: 0.7342 - val_accuracy: 0.9219
Epoch 45/50
136/136 [==============================] - 1s 5ms/step - loss: 0.1556 - accuracy: 0.9508 - val_loss: 0.9154 - val_accuracy: 0.9219
Epoch 46/50
136/136 [==============================] - 1s 5ms/step - loss: 0.1721 - accuracy: 0.9504 - val_loss: 0.8006 - val_accuracy: 0.9164
Epoch 47/50

Epoch 47/50
155/155 [==============================] - 1s 5ms/step - loss: 0.1296 - accuracy: 0.9613 - val_loss: 0.7061 - val_accuracy: 0.9395
Epoch 48/50
155/155 [==============================] - 1s 5ms/step - loss: 0.1224 - accuracy: 0.9657 - val_loss: 0.6697 - val_accuracy: 0.9315
Epoch 49/50
155/155 [==============================] - 1s 5ms/step - loss: 0.1065 - accuracy: 0.9671 - val_loss: 0.7106 - val_accuracy: 0.9411
Epoch 50/50
155/155 [==============================] - 1s 5ms/step - loss: 0.1030 - accuracy: 0.9675 - val_loss: 0.7637 - val_accuracy: 0.9371
Model f_8 generated

3/3 [==============================] - 0s 22ms/step - loss: 10.7187 - accuracy: 0.8436
Epoch 1/50
139/139 [==============================] - 1s 6ms/step - loss: 22.9437 - accuracy: 0.5508 - val_loss: 3.9222 - val_accuracy: 0.7622
Epoch 2/50
139/139 [==============================] - 1s 5ms/step - loss: 4.2291 - accuracy: 0.6790 - val_loss: 1.9710 - val_accuracy: 0.7658
Epoch 3/50
139/139 [===============

Epoch 3/50
156/156 [==============================] - 1s 5ms/step - loss: 1.5010 - accuracy: 0.6005 - val_loss: 1.2547 - val_accuracy: 0.7253
Epoch 4/50
 35/156 [=====>........................] - ETA: 0s - loss: 1.2147 - accuracy: 0.6348

In [58]:
run_experiment("MNIST", "hetero-dir", 10, 0.5, 0.0003, 50) 
# [0.670199990272522,
# 0.8228999972343445,
# 0.7373999953269958,
# 0.7164000272750854,
# 0.6353999972343445,
# 0.8159999847412109,
# 0.661300003528595,
# 0.6909999847412109,
# 0.8111000061035156,
# 0.676800012588501]

Epoch 1/50
115/115 [==============================] - 2s 8ms/step - loss: 15.4815 - accuracy: 0.8060 - val_loss: 3.1210 - val_accuracy: 0.9195
Epoch 2/50
115/115 [==============================] - 1s 4ms/step - loss: 2.6397 - accuracy: 0.9102 - val_loss: 2.4788 - val_accuracy: 0.9227
Epoch 3/50
115/115 [==============================] - 1s 4ms/step - loss: 1.4519 - accuracy: 0.9298 - val_loss: 1.8115 - val_accuracy: 0.9358
Epoch 4/50
115/115 [==============================] - 1s 5ms/step - loss: 0.8311 - accuracy: 0.9421 - val_loss: 1.5869 - val_accuracy: 0.9358
Epoch 5/50
115/115 [==============================] - 0s 4ms/step - loss: 0.6552 - accuracy: 0.9489 - val_loss: 1.5541 - val_accuracy: 0.9347
Epoch 6/50
115/115 [==============================] - 0s 4ms/step - loss: 0.3970 - accuracy: 0.9600 - val_loss: 1.3738 - val_accuracy: 0.9347
Epoch 7/50
115/115 [==============================] - 0s 4ms/step - loss: 0.3566 - accuracy: 0.9524 - val_loss: 1.1544 - val_accuracy: 0.9423
Epoch

Epoch 8/50
184/184 [==============================] - 1s 6ms/step - loss: 0.5327 - accuracy: 0.8545 - val_loss: 0.6592 - val_accuracy: 0.8832
Epoch 9/50
184/184 [==============================] - 1s 5ms/step - loss: 0.4786 - accuracy: 0.8585 - val_loss: 0.6147 - val_accuracy: 0.8927
Epoch 10/50
184/184 [==============================] - 1s 4ms/step - loss: 0.4392 - accuracy: 0.8761 - val_loss: 0.6229 - val_accuracy: 0.9069
Epoch 11/50
184/184 [==============================] - 1s 5ms/step - loss: 0.4172 - accuracy: 0.8804 - val_loss: 0.5549 - val_accuracy: 0.8995
Epoch 12/50
184/184 [==============================] - 1s 4ms/step - loss: 0.4275 - accuracy: 0.8785 - val_loss: 0.5981 - val_accuracy: 0.9069
Epoch 13/50
184/184 [==============================] - 1s 5ms/step - loss: 0.3871 - accuracy: 0.8902 - val_loss: 0.5555 - val_accuracy: 0.9062
Epoch 14/50
184/184 [==============================] - 1s 4ms/step - loss: 0.3535 - accuracy: 0.8989 - val_loss: 0.4896 - val_accuracy: 0.9130
E

Epoch 15/50
180/180 [==============================] - 1s 5ms/step - loss: 0.2241 - accuracy: 0.9428 - val_loss: 0.4226 - val_accuracy: 0.9394
Epoch 16/50
180/180 [==============================] - 1s 5ms/step - loss: 0.2062 - accuracy: 0.9442 - val_loss: 0.4355 - val_accuracy: 0.9387
Epoch 17/50
180/180 [==============================] - 1s 4ms/step - loss: 0.2001 - accuracy: 0.9516 - val_loss: 0.4752 - val_accuracy: 0.9491
Epoch 18/50
180/180 [==============================] - 1s 5ms/step - loss: 0.1876 - accuracy: 0.9482 - val_loss: 0.4009 - val_accuracy: 0.9443
Epoch 19/50
180/180 [==============================] - 1s 5ms/step - loss: 0.1861 - accuracy: 0.9514 - val_loss: 0.4463 - val_accuracy: 0.9526
Epoch 20/50
180/180 [==============================] - 1s 4ms/step - loss: 0.1628 - accuracy: 0.9547 - val_loss: 0.3780 - val_accuracy: 0.9470
Epoch 21/50
180/180 [==============================] - 1s 4ms/step - loss: 0.1945 - accuracy: 0.9507 - val_loss: 0.4256 - val_accuracy: 0.9436

Epoch 22/50
165/165 [==============================] - 1s 4ms/step - loss: 0.2064 - accuracy: 0.9466 - val_loss: 0.6163 - val_accuracy: 0.9271
Epoch 23/50
165/165 [==============================] - 1s 4ms/step - loss: 0.1775 - accuracy: 0.9478 - val_loss: 0.8062 - val_accuracy: 0.9202
Epoch 24/50
165/165 [==============================] - 1s 5ms/step - loss: 0.2028 - accuracy: 0.9421 - val_loss: 0.5781 - val_accuracy: 0.9354
Epoch 25/50
165/165 [==============================] - 1s 6ms/step - loss: 0.2110 - accuracy: 0.9462 - val_loss: 0.7263 - val_accuracy: 0.9233
Epoch 26/50
165/165 [==============================] - 1s 4ms/step - loss: 0.2055 - accuracy: 0.9430 - val_loss: 0.5600 - val_accuracy: 0.9331
Epoch 27/50
165/165 [==============================] - 1s 4ms/step - loss: 0.1728 - accuracy: 0.9516 - val_loss: 0.6089 - val_accuracy: 0.9415
Epoch 28/50
165/165 [==============================] - 1s 5ms/step - loss: 0.1804 - accuracy: 0.9521 - val_loss: 1.0307 - val_accuracy: 0.9248

Epoch 29/50
151/151 [==============================] - 1s 4ms/step - loss: 0.1434 - accuracy: 0.9731 - val_loss: 0.6145 - val_accuracy: 0.9594
Epoch 30/50
151/151 [==============================] - 1s 4ms/step - loss: 0.1249 - accuracy: 0.9712 - val_loss: 0.4519 - val_accuracy: 0.9569
Epoch 31/50
151/151 [==============================] - 1s 4ms/step - loss: 0.1340 - accuracy: 0.9741 - val_loss: 0.4766 - val_accuracy: 0.9544
Epoch 32/50
151/151 [==============================] - 1s 4ms/step - loss: 0.0938 - accuracy: 0.9791 - val_loss: 0.4343 - val_accuracy: 0.9644
Epoch 33/50
151/151 [==============================] - 1s 4ms/step - loss: 0.1164 - accuracy: 0.9766 - val_loss: 0.4043 - val_accuracy: 0.9685
Epoch 34/50
151/151 [==============================] - 1s 4ms/step - loss: 0.0672 - accuracy: 0.9824 - val_loss: 0.4264 - val_accuracy: 0.9594
Epoch 35/50
151/151 [==============================] - 1s 4ms/step - loss: 0.0954 - accuracy: 0.9776 - val_loss: 0.4834 - val_accuracy: 0.9594

Epoch 36/50
113/113 [==============================] - 0s 4ms/step - loss: 0.2031 - accuracy: 0.9320 - val_loss: 0.8250 - val_accuracy: 0.9157
Epoch 37/50
113/113 [==============================] - 0s 4ms/step - loss: 0.2108 - accuracy: 0.9337 - val_loss: 0.8575 - val_accuracy: 0.9224
Epoch 38/50
113/113 [==============================] - 0s 4ms/step - loss: 0.1817 - accuracy: 0.9376 - val_loss: 0.6948 - val_accuracy: 0.9257
Epoch 39/50
113/113 [==============================] - 0s 4ms/step - loss: 0.1753 - accuracy: 0.9348 - val_loss: 0.6468 - val_accuracy: 0.9213
Epoch 40/50
113/113 [==============================] - 0s 4ms/step - loss: 0.1855 - accuracy: 0.9406 - val_loss: 0.6903 - val_accuracy: 0.9257
Epoch 41/50
113/113 [==============================] - 0s 4ms/step - loss: 0.2290 - accuracy: 0.9329 - val_loss: 0.5547 - val_accuracy: 0.9091
Epoch 42/50
113/113 [==============================] - 0s 4ms/step - loss: 0.2061 - accuracy: 0.9342 - val_loss: 0.7482 - val_accuracy: 0.9146

99/99 [==============================] - 0s 4ms/step - loss: 0.1161 - accuracy: 0.9751 - val_loss: 0.6049 - val_accuracy: 0.9495
Epoch 44/50
99/99 [==============================] - 0s 5ms/step - loss: 0.1225 - accuracy: 0.9668 - val_loss: 0.6005 - val_accuracy: 0.9470
Epoch 45/50
99/99 [==============================] - 0s 4ms/step - loss: 0.1060 - accuracy: 0.9694 - val_loss: 0.6236 - val_accuracy: 0.9495
Epoch 46/50
99/99 [==============================] - 0s 4ms/step - loss: 0.0953 - accuracy: 0.9728 - val_loss: 0.6577 - val_accuracy: 0.9470
Epoch 47/50
99/99 [==============================] - 0s 5ms/step - loss: 0.1080 - accuracy: 0.9691 - val_loss: 0.6091 - val_accuracy: 0.9444
Epoch 48/50
99/99 [==============================] - 1s 5ms/step - loss: 0.1094 - accuracy: 0.9700 - val_loss: 0.5837 - val_accuracy: 0.9457
Epoch 49/50
99/99 [==============================] - 1s 5ms/step - loss: 0.0977 - accuracy: 0.9766 - val_loss: 0.6400 - val_accuracy: 0.9432
Epoch 50/50
3/3 [========

Epoch 50/50
3/3 [==============================] - 0s 21ms/step - loss: 14.8422 - accuracy: 0.6910
Epoch 1/50
172/172 [==============================] - 2s 6ms/step - loss: 17.8538 - accuracy: 0.6494 - val_loss: 2.5540 - val_accuracy: 0.8455
Epoch 2/50
172/172 [==============================] - 1s 5ms/step - loss: 2.7053 - accuracy: 0.7200 - val_loss: 1.0911 - val_accuracy: 0.8010
Epoch 3/50
172/172 [==============================] - 1s 5ms/step - loss: 1.2755 - accuracy: 0.7362 - val_loss: 0.9095 - val_accuracy: 0.7945
Epoch 4/50
172/172 [==============================] - 1s 4ms/step - loss: 0.9510 - accuracy: 0.7655 - val_loss: 0.7569 - val_accuracy: 0.8579
Epoch 5/50
172/172 [==============================] - 1s 4ms/step - loss: 0.7779 - accuracy: 0.8018 - val_loss: 0.6847 - val_accuracy: 0.8601
Epoch 6/50
172/172 [==============================] - 1s 5ms/step - loss: 0.6299 - accuracy: 0.8246 - val_loss: 0.7008 - val_accuracy: 0.8710
Epoch 7/50
172/172 [============================

173/173 [==============================] - 1s 4ms/step - loss: 0.3746 - accuracy: 0.9360 - val_loss: 0.4805 - val_accuracy: 0.9508
Epoch 7/50
173/173 [==============================] - 1s 5ms/step - loss: 0.3109 - accuracy: 0.9414 - val_loss: 0.5031 - val_accuracy: 0.9458
Epoch 8/50
173/173 [==============================] - 1s 5ms/step - loss: 0.2574 - accuracy: 0.9512 - val_loss: 0.3717 - val_accuracy: 0.9566
Epoch 9/50
173/173 [==============================] - 1s 4ms/step - loss: 0.2483 - accuracy: 0.9492 - val_loss: 0.4068 - val_accuracy: 0.9588
Epoch 10/50
173/173 [==============================] - 1s 4ms/step - loss: 0.2356 - accuracy: 0.9515 - val_loss: 0.4596 - val_accuracy: 0.9559
Epoch 11/50
173/173 [==============================] - 1s 4ms/step - loss: 0.1834 - accuracy: 0.9616 - val_loss: 0.4385 - val_accuracy: 0.9617
Epoch 12/50
173/173 [==============================] - 1s 4ms/step - loss: 0.1813 - accuracy: 0.9602 - val_loss: 0.4102 - val_accuracy: 0.9624
Epoch 13/50
17

[0.670199990272522,
 0.8228999972343445,
 0.7373999953269958,
 0.7164000272750854,
 0.6353999972343445,
 0.8159999847412109,
 0.661300003528595,
 0.6909999847412109,
 0.8111000061035156,
 0.676800012588501]

In [60]:
run_experiment("MNIST", "hetero-dir", 10, 100, 0.0003, 50) 
# [0.9225000143051147,
# 0.9221000075340271,
# 0.9243999719619751,
# 0.9186000227928162,
# 0.9334999918937683,
# 0.9290000200271606,
# 0.9225000143051147,
# 0.9172000288963318,
# 0.9314000010490417,
# 0.9336000084877014]

Epoch 1/50
160/160 [==============================] - 2s 7ms/step - loss: 26.2955 - accuracy: 0.4840 - val_loss: 3.8166 - val_accuracy: 0.7203
Epoch 2/50
160/160 [==============================] - 1s 4ms/step - loss: 3.6623 - accuracy: 0.5908 - val_loss: 1.7549 - val_accuracy: 0.6508
Epoch 3/50
160/160 [==============================] - 1s 4ms/step - loss: 1.7492 - accuracy: 0.5854 - val_loss: 1.4961 - val_accuracy: 0.6953
Epoch 4/50
160/160 [==============================] - 1s 4ms/step - loss: 1.3489 - accuracy: 0.6383 - val_loss: 1.3022 - val_accuracy: 0.7398
Epoch 5/50
160/160 [==============================] - 1s 4ms/step - loss: 1.1691 - accuracy: 0.6832 - val_loss: 1.1547 - val_accuracy: 0.7703
Epoch 6/50
160/160 [==============================] - 1s 4ms/step - loss: 0.9587 - accuracy: 0.7195 - val_loss: 1.0085 - val_accuracy: 0.8086
Epoch 7/50
160/160 [==============================] - 1s 4ms/step - loss: 0.8360 - accuracy: 0.7527 - val_loss: 1.0277 - val_accuracy: 0.7969
Epoch

Epoch 8/50
146/146 [==============================] - 1s 5ms/step - loss: 0.7041 - accuracy: 0.7995 - val_loss: 0.6982 - val_accuracy: 0.8553
Epoch 9/50
146/146 [==============================] - 1s 5ms/step - loss: 0.6343 - accuracy: 0.8113 - val_loss: 0.7261 - val_accuracy: 0.8536
Epoch 10/50
146/146 [==============================] - 1s 5ms/step - loss: 0.5875 - accuracy: 0.8306 - val_loss: 0.6942 - val_accuracy: 0.8733
Epoch 11/50
146/146 [==============================] - 1s 4ms/step - loss: 0.5287 - accuracy: 0.8441 - val_loss: 0.6777 - val_accuracy: 0.8690
Epoch 12/50
146/146 [==============================] - 1s 4ms/step - loss: 0.4709 - accuracy: 0.8552 - val_loss: 0.6209 - val_accuracy: 0.8887
Epoch 13/50
146/146 [==============================] - 1s 4ms/step - loss: 0.4795 - accuracy: 0.8616 - val_loss: 0.6491 - val_accuracy: 0.8741
Epoch 14/50
146/146 [==============================] - 1s 4ms/step - loss: 0.4522 - accuracy: 0.8655 - val_loss: 0.7158 - val_accuracy: 0.8904
E

Epoch 15/50
143/143 [==============================] - 1s 4ms/step - loss: 0.5368 - accuracy: 0.8385 - val_loss: 0.5422 - val_accuracy: 0.8734
Epoch 16/50
143/143 [==============================] - 1s 4ms/step - loss: 0.4657 - accuracy: 0.8526 - val_loss: 0.5387 - val_accuracy: 0.8795
Epoch 17/50
143/143 [==============================] - 1s 4ms/step - loss: 0.4556 - accuracy: 0.8557 - val_loss: 0.5174 - val_accuracy: 0.8760
Epoch 18/50
143/143 [==============================] - 1s 5ms/step - loss: 0.4061 - accuracy: 0.8713 - val_loss: 0.5116 - val_accuracy: 0.8874
Epoch 19/50
143/143 [==============================] - 1s 5ms/step - loss: 0.4194 - accuracy: 0.8741 - val_loss: 0.4999 - val_accuracy: 0.9059
Epoch 20/50
143/143 [==============================] - 1s 5ms/step - loss: 0.4274 - accuracy: 0.8669 - val_loss: 0.5406 - val_accuracy: 0.8883
Epoch 21/50
143/143 [==============================] - 1s 5ms/step - loss: 0.4410 - accuracy: 0.8807 - val_loss: 0.6832 - val_accuracy: 0.8813

Epoch 22/50
147/147 [==============================] - 1s 4ms/step - loss: 0.3517 - accuracy: 0.9051 - val_loss: 0.6797 - val_accuracy: 0.8960
Epoch 23/50
147/147 [==============================] - 1s 4ms/step - loss: 0.3333 - accuracy: 0.9002 - val_loss: 0.6346 - val_accuracy: 0.9045
Epoch 24/50
147/147 [==============================] - 1s 4ms/step - loss: 0.3252 - accuracy: 0.9072 - val_loss: 0.8084 - val_accuracy: 0.8934
Epoch 25/50
147/147 [==============================] - 1s 5ms/step - loss: 0.3171 - accuracy: 0.9057 - val_loss: 0.6929 - val_accuracy: 0.9020
Epoch 26/50
147/147 [==============================] - 1s 4ms/step - loss: 0.3085 - accuracy: 0.9047 - val_loss: 0.6842 - val_accuracy: 0.9020
Epoch 27/50
147/147 [==============================] - 1s 4ms/step - loss: 0.3078 - accuracy: 0.9145 - val_loss: 0.5990 - val_accuracy: 0.9037
Epoch 28/50
147/147 [==============================] - 1s 4ms/step - loss: 0.2642 - accuracy: 0.9198 - val_loss: 0.5655 - val_accuracy: 0.9190

Epoch 29/50
152/152 [==============================] - 1s 5ms/step - loss: 0.3047 - accuracy: 0.9073 - val_loss: 0.5709 - val_accuracy: 0.9149
Epoch 30/50
152/152 [==============================] - 1s 6ms/step - loss: 0.2820 - accuracy: 0.9090 - val_loss: 0.6386 - val_accuracy: 0.9059
Epoch 31/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2690 - accuracy: 0.9172 - val_loss: 0.6091 - val_accuracy: 0.9158
Epoch 32/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2832 - accuracy: 0.9162 - val_loss: 0.5194 - val_accuracy: 0.9182
Epoch 33/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2707 - accuracy: 0.9166 - val_loss: 0.6739 - val_accuracy: 0.9133
Epoch 34/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2727 - accuracy: 0.9158 - val_loss: 0.6083 - val_accuracy: 0.9133
Epoch 35/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2515 - accuracy: 0.9228 - val_loss: 0.5655 - val_accuracy: 0.9174

Epoch 36/50
144/144 [==============================] - 1s 4ms/step - loss: 0.2844 - accuracy: 0.9107 - val_loss: 0.7346 - val_accuracy: 0.9162
Epoch 37/50
144/144 [==============================] - 1s 4ms/step - loss: 0.3063 - accuracy: 0.9098 - val_loss: 0.6122 - val_accuracy: 0.9074
Epoch 38/50
144/144 [==============================] - 1s 4ms/step - loss: 0.2648 - accuracy: 0.9113 - val_loss: 0.8944 - val_accuracy: 0.9118
Epoch 39/50
144/144 [==============================] - 1s 4ms/step - loss: 0.2628 - accuracy: 0.9161 - val_loss: 0.6725 - val_accuracy: 0.9205
Epoch 40/50
144/144 [==============================] - 1s 4ms/step - loss: 0.2879 - accuracy: 0.9059 - val_loss: 0.5308 - val_accuracy: 0.9214
Epoch 41/50
144/144 [==============================] - 1s 4ms/step - loss: 0.2772 - accuracy: 0.9185 - val_loss: 0.6613 - val_accuracy: 0.9170
Epoch 42/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2781 - accuracy: 0.9185 - val_loss: 0.6309 - val_accuracy: 0.9162

Epoch 43/50
153/153 [==============================] - 1s 4ms/step - loss: 0.1873 - accuracy: 0.9442 - val_loss: 0.9393 - val_accuracy: 0.9172
Epoch 44/50
153/153 [==============================] - 1s 5ms/step - loss: 0.2250 - accuracy: 0.9371 - val_loss: 1.0153 - val_accuracy: 0.9066
Epoch 45/50
153/153 [==============================] - 1s 4ms/step - loss: 0.2457 - accuracy: 0.9319 - val_loss: 0.8607 - val_accuracy: 0.9189
Epoch 46/50
153/153 [==============================] - 1s 4ms/step - loss: 0.2099 - accuracy: 0.9405 - val_loss: 0.8659 - val_accuracy: 0.9221
Epoch 47/50
153/153 [==============================] - 1s 4ms/step - loss: 0.1820 - accuracy: 0.9465 - val_loss: 0.8328 - val_accuracy: 0.9320
Epoch 48/50
153/153 [==============================] - 1s 4ms/step - loss: 0.1890 - accuracy: 0.9487 - val_loss: 0.9347 - val_accuracy: 0.9205
Epoch 49/50
153/153 [==============================] - 1s 4ms/step - loss: 0.1723 - accuracy: 0.9510 - val_loss: 0.9355 - val_accuracy: 0.9139

Epoch 50/50
3/3 [==============================] - 0s 22ms/step - loss: 0.7800 - accuracy: 0.9172
Epoch 1/50
149/149 [==============================] - 1s 6ms/step - loss: 25.0089 - accuracy: 0.4964 - val_loss: 4.0124 - val_accuracy: 0.7159
Epoch 2/50
149/149 [==============================] - 1s 5ms/step - loss: 3.5498 - accuracy: 0.6179 - val_loss: 1.5787 - val_accuracy: 0.6585
Epoch 3/50
149/149 [==============================] - 1s 4ms/step - loss: 1.7160 - accuracy: 0.6006 - val_loss: 1.2869 - val_accuracy: 0.7074
Epoch 4/50
149/149 [==============================] - 1s 5ms/step - loss: 1.2647 - accuracy: 0.6624 - val_loss: 1.1318 - val_accuracy: 0.7378
Epoch 5/50
149/149 [==============================] - 1s 5ms/step - loss: 1.0784 - accuracy: 0.6960 - val_loss: 1.0906 - val_accuracy: 0.7723
Epoch 6/50
149/149 [==============================] - 1s 5ms/step - loss: 0.9381 - accuracy: 0.7238 - val_loss: 0.9995 - val_accuracy: 0.8120
Epoch 7/50
149/149 [=============================

156/156 [==============================] - 1s 6ms/step - loss: 0.9040 - accuracy: 0.7471 - val_loss: 0.8931 - val_accuracy: 0.8047
Epoch 7/50
156/156 [==============================] - 1s 6ms/step - loss: 0.8096 - accuracy: 0.7732 - val_loss: 0.8585 - val_accuracy: 0.8183
Epoch 8/50
156/156 [==============================] - 1s 5ms/step - loss: 0.7322 - accuracy: 0.7883 - val_loss: 0.8949 - val_accuracy: 0.8167
Epoch 9/50
156/156 [==============================] - 1s 6ms/step - loss: 0.6708 - accuracy: 0.8080 - val_loss: 0.8580 - val_accuracy: 0.8248
Epoch 10/50
156/156 [==============================] - 1s 6ms/step - loss: 0.5970 - accuracy: 0.8193 - val_loss: 0.8274 - val_accuracy: 0.8497
Epoch 11/50
156/156 [==============================] - 1s 6ms/step - loss: 0.5787 - accuracy: 0.8225 - val_loss: 0.8155 - val_accuracy: 0.8408
Epoch 12/50
156/156 [==============================] - 1s 5ms/step - loss: 0.5550 - accuracy: 0.8404 - val_loss: 0.8372 - val_accuracy: 0.8513
Epoch 13/50
15

[0.9225000143051147,
 0.9221000075340271,
 0.9243999719619751,
 0.9186000227928162,
 0.9334999918937683,
 0.9290000200271606,
 0.9225000143051147,
 0.9172000288963318,
 0.9314000010490417,
 0.9336000084877014]

In [64]:
run_experiment("MNIST", "hetero-dir", 10, 10, 0.0003, 50)

Epoch 1/50
143/143 [==============================] - 2s 10ms/step - loss: 30.1909 - accuracy: 0.5116 - val_loss: 3.4657 - val_accuracy: 0.7937
Epoch 2/50
143/143 [==============================] - 1s 5ms/step - loss: 5.3340 - accuracy: 0.6752 - val_loss: 1.8515 - val_accuracy: 0.8068
Epoch 3/50
143/143 [==============================] - 1s 5ms/step - loss: 2.4844 - accuracy: 0.6874 - val_loss: 1.3488 - val_accuracy: 0.7735
Epoch 4/50
143/143 [==============================] - 1s 5ms/step - loss: 1.6161 - accuracy: 0.6734 - val_loss: 1.0566 - val_accuracy: 0.7709
Epoch 5/50
143/143 [==============================] - 1s 5ms/step - loss: 1.1447 - accuracy: 0.7252 - val_loss: 0.9643 - val_accuracy: 0.7946
Epoch 6/50
143/143 [==============================] - 1s 5ms/step - loss: 0.9653 - accuracy: 0.7491 - val_loss: 0.8533 - val_accuracy: 0.8297
Epoch 7/50
143/143 [==============================] - 1s 5ms/step - loss: 0.8303 - accuracy: 0.7737 - val_loss: 0.7813 - val_accuracy: 0.8218
Epoc

161/161 [==============================] - 1s 4ms/step - loss: 0.7836 - accuracy: 0.7565 - val_loss: 0.9588 - val_accuracy: 0.8043
Epoch 8/50
161/161 [==============================] - 1s 4ms/step - loss: 0.6852 - accuracy: 0.7874 - val_loss: 0.9418 - val_accuracy: 0.8261
Epoch 9/50
161/161 [==============================] - 1s 4ms/step - loss: 0.6527 - accuracy: 0.8017 - val_loss: 0.8966 - val_accuracy: 0.8307
Epoch 10/50
161/161 [==============================] - 1s 4ms/step - loss: 0.6210 - accuracy: 0.8111 - val_loss: 0.9216 - val_accuracy: 0.8416
Epoch 11/50
161/161 [==============================] - 1s 5ms/step - loss: 0.5660 - accuracy: 0.8239 - val_loss: 0.9465 - val_accuracy: 0.8339
Epoch 12/50
161/161 [==============================] - 1s 5ms/step - loss: 0.5286 - accuracy: 0.8357 - val_loss: 0.8187 - val_accuracy: 0.8595
Epoch 13/50
161/161 [==============================] - 1s 5ms/step - loss: 0.4880 - accuracy: 0.8429 - val_loss: 0.8940 - val_accuracy: 0.8634
Epoch 14/50
1

Epoch 14/50
168/168 [==============================] - 1s 4ms/step - loss: 0.4169 - accuracy: 0.8796 - val_loss: 0.6679 - val_accuracy: 0.8714
Epoch 15/50
168/168 [==============================] - 1s 4ms/step - loss: 0.4167 - accuracy: 0.8731 - val_loss: 0.7302 - val_accuracy: 0.8782
Epoch 16/50
168/168 [==============================] - 1s 4ms/step - loss: 0.4087 - accuracy: 0.8729 - val_loss: 0.7179 - val_accuracy: 0.8737
Epoch 17/50
168/168 [==============================] - 1s 4ms/step - loss: 0.3720 - accuracy: 0.8858 - val_loss: 0.7098 - val_accuracy: 0.8797
Epoch 18/50
168/168 [==============================] - 1s 4ms/step - loss: 0.3664 - accuracy: 0.8884 - val_loss: 0.7139 - val_accuracy: 0.8819
Epoch 19/50
168/168 [==============================] - 1s 4ms/step - loss: 0.3773 - accuracy: 0.8955 - val_loss: 0.7202 - val_accuracy: 0.8789
Epoch 20/50
168/168 [==============================] - 1s 4ms/step - loss: 0.3403 - accuracy: 0.8953 - val_loss: 0.6040 - val_accuracy: 0.8954

144/144 [==============================] - 1s 5ms/step - loss: 0.3157 - accuracy: 0.9090 - val_loss: 0.6688 - val_accuracy: 0.8935
Epoch 21/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2668 - accuracy: 0.9166 - val_loss: 0.6040 - val_accuracy: 0.8918
Epoch 22/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2421 - accuracy: 0.9297 - val_loss: 0.6401 - val_accuracy: 0.9023
Epoch 23/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2483 - accuracy: 0.9223 - val_loss: 0.6580 - val_accuracy: 0.8944
Epoch 24/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2525 - accuracy: 0.9300 - val_loss: 0.6940 - val_accuracy: 0.8962
Epoch 25/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2527 - accuracy: 0.9234 - val_loss: 0.7132 - val_accuracy: 0.8997
Epoch 26/50
144/144 [==============================] - 1s 5ms/step - loss: 0.2602 - accuracy: 0.9269 - val_loss: 0.6272 - val_accuracy: 0.9127
Epoch 27/50

Epoch 27/50
155/155 [==============================] - 1s 5ms/step - loss: 0.3203 - accuracy: 0.8990 - val_loss: 1.0124 - val_accuracy: 0.8821
Epoch 28/50
155/155 [==============================] - 1s 5ms/step - loss: 0.3108 - accuracy: 0.8994 - val_loss: 0.8630 - val_accuracy: 0.8877
Epoch 29/50
155/155 [==============================] - 1s 5ms/step - loss: 0.2782 - accuracy: 0.9126 - val_loss: 0.8267 - val_accuracy: 0.9087
Epoch 30/50
155/155 [==============================] - 1s 5ms/step - loss: 0.2880 - accuracy: 0.9116 - val_loss: 0.8159 - val_accuracy: 0.9031
Epoch 31/50
155/155 [==============================] - 1s 5ms/step - loss: 0.2796 - accuracy: 0.9132 - val_loss: 0.9020 - val_accuracy: 0.8877
Epoch 32/50
155/155 [==============================] - 1s 5ms/step - loss: 0.2863 - accuracy: 0.9138 - val_loss: 0.8629 - val_accuracy: 0.9103
Epoch 33/50
155/155 [==============================] - 1s 5ms/step - loss: 0.2755 - accuracy: 0.9180 - val_loss: 0.8501 - val_accuracy: 0.9071

153/153 [==============================] - 1s 5ms/step - loss: 0.1569 - accuracy: 0.9506 - val_loss: 0.4362 - val_accuracy: 0.9476
Epoch 34/50
153/153 [==============================] - 1s 5ms/step - loss: 0.1551 - accuracy: 0.9551 - val_loss: 0.4415 - val_accuracy: 0.9525
Epoch 35/50
153/153 [==============================] - 1s 5ms/step - loss: 0.1272 - accuracy: 0.9619 - val_loss: 0.4570 - val_accuracy: 0.9533
Epoch 36/50
153/153 [==============================] - 1s 5ms/step - loss: 0.1494 - accuracy: 0.9582 - val_loss: 0.4260 - val_accuracy: 0.9517
Epoch 37/50
153/153 [==============================] - 1s 5ms/step - loss: 0.1695 - accuracy: 0.9562 - val_loss: 0.4715 - val_accuracy: 0.9435
Epoch 38/50
153/153 [==============================] - 1s 5ms/step - loss: 0.1434 - accuracy: 0.9560 - val_loss: 0.4782 - val_accuracy: 0.9484
Epoch 39/50
153/153 [==============================] - 1s 5ms/step - loss: 0.1779 - accuracy: 0.9539 - val_loss: 0.5813 - val_accuracy: 0.9419
Epoch 40/50

152/152 [==============================] - 1s 5ms/step - loss: 0.2020 - accuracy: 0.9387 - val_loss: 0.9813 - val_accuracy: 0.9010
Epoch 40/50
152/152 [==============================] - 1s 6ms/step - loss: 0.2308 - accuracy: 0.9327 - val_loss: 0.8697 - val_accuracy: 0.8952
Epoch 41/50
152/152 [==============================] - 1s 5ms/step - loss: 0.1894 - accuracy: 0.9381 - val_loss: 0.8833 - val_accuracy: 0.9084
Epoch 42/50
152/152 [==============================] - 1s 5ms/step - loss: 0.1855 - accuracy: 0.9416 - val_loss: 1.0307 - val_accuracy: 0.9026
Epoch 43/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2008 - accuracy: 0.9414 - val_loss: 0.8743 - val_accuracy: 0.9010
Epoch 44/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2097 - accuracy: 0.9439 - val_loss: 1.0291 - val_accuracy: 0.9002
Epoch 45/50
152/152 [==============================] - 1s 5ms/step - loss: 0.2549 - accuracy: 0.9373 - val_loss: 0.7421 - val_accuracy: 0.9026
Epoch 46/50

Epoch 46/50
142/142 [==============================] - 1s 5ms/step - loss: 0.1800 - accuracy: 0.9463 - val_loss: 0.6432 - val_accuracy: 0.9293
Epoch 47/50
142/142 [==============================] - 1s 5ms/step - loss: 0.2145 - accuracy: 0.9410 - val_loss: 0.6589 - val_accuracy: 0.9302
Epoch 48/50
142/142 [==============================] - 1s 5ms/step - loss: 0.2793 - accuracy: 0.9335 - val_loss: 0.6188 - val_accuracy: 0.9302
Epoch 49/50
142/142 [==============================] - 1s 5ms/step - loss: 0.1613 - accuracy: 0.9476 - val_loss: 0.7315 - val_accuracy: 0.9170
Epoch 50/50
142/142 [==============================] - 1s 5ms/step - loss: 0.1639 - accuracy: 0.9476 - val_loss: 0.6594 - val_accuracy: 0.9284
Model f_7 generated

3/3 [==============================] - 0s 22ms/step - loss: 0.6644 - accuracy: 0.9295
Epoch 1/50
148/148 [==============================] - 1s 6ms/step - loss: 25.4179 - accuracy: 0.5288 - val_loss: 4.2031 - val_accuracy: 0.7523
Epoch 2/50
148/148 [===============

Epoch 2/50
139/139 [==============================] - 1s 5ms/step - loss: 3.6606 - accuracy: 0.6414 - val_loss: 1.7315 - val_accuracy: 0.7046
Epoch 3/50
139/139 [==============================] - 1s 5ms/step - loss: 1.7456 - accuracy: 0.6168 - val_loss: 1.3738 - val_accuracy: 0.7055
Epoch 4/50
139/139 [==============================] - 1s 5ms/step - loss: 1.2826 - accuracy: 0.6441 - val_loss: 1.2385 - val_accuracy: 0.7272
Epoch 5/50
139/139 [==============================] - 1s 5ms/step - loss: 1.0734 - accuracy: 0.6952 - val_loss: 1.1246 - val_accuracy: 0.7380
Epoch 6/50
139/139 [==============================] - 1s 5ms/step - loss: 0.9474 - accuracy: 0.7122 - val_loss: 1.1141 - val_accuracy: 0.7733
Epoch 7/50
139/139 [==============================] - 1s 5ms/step - loss: 0.8541 - accuracy: 0.7427 - val_loss: 1.0280 - val_accuracy: 0.7940
Epoch 8/50
139/139 [==============================] - 1s 6ms/step - loss: 0.7403 - accuracy: 0.7714 - val_loss: 0.9817 - val_accuracy: 0.8229
Epoch 

[0.9182999730110168,
 0.9240000247955322,
 0.9269999861717224,
 0.9222000241279602,
 0.9175000190734863,
 0.7648000121116638,
 0.9240000247955322,
 0.9294999837875366,
 0.9218000173568726,
 0.9110999703407288]

In [40]:
subset_map = partition("MNIST", "hetero-dir", 10, 0.5)
train, test = get_data("MNIST")

In [48]:
train[1][subset_map[0]]

tensor([6, 6, 0,  ..., 0, 5, 0], dtype=torch.uint8)

In [50]:
train[1][[22017,18904]]

tensor([6, 6], dtype=torch.uint8)

In [ ]:
dataset_loaders = {
    "MNIST": mnist.load_data,
    "CIFAR10": cifar10.load_data,
}

dataset_type = "MNISt"
try:
    train, test = dataset_loaders[dataset_type]
except KeyError:
    raise ValueError(f"Invalid mode: {dataset_type}")

In [ ]:
[0.670199990272522,
 0.8228999972343445,
 0.7373999953269958,
 0.7164000272750854,
 0.6353999972343445,
 0.8159999847412109,
 0.661300003528595,
 0.6909999847412109,
 0.8111000061035156,
 0.676800012588501]